# Mining Alpha — 单因子诊断深挖

输入一个 alpha 号，输出满眼诊断：
1. **公式与说明** — 来自 `_ALPHA_REGISTRY`
2. **滚动 IC 时间序列** — 看因子在不同时段的稳定性
3. **IC 自相关** — 信号衰减速度 (lag 1-10)
4. **Top decile 累计超额** — 实际选股力的 PnL 等价物
5. **横截面分布** — 因子值在最新一日的分布（直方图 + 长尾）
6. **NaN 率随时间** — 数据/算子健康度


In [ ]:
ALPHA_NUM = 22  # ← 改这里换不同 alpha
HORIZON = 5

In [ ]:
import sys
from pathlib import Path
BACKEND = Path('..').resolve()
sys.path.insert(0, str(BACKEND))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from mining_alpha.synthetic_demo import generate_synthetic_panel
from mining_alpha.alpha191_factors import _ALPHA_REGISTRY, compute_alpha
from mining_alpha.ic_report import compute_forward_return, daily_ic, top_decile_excess
from mining_alpha.preprocess import preprocess_pipeline

## 1. 公式与说明

In [ ]:
if ALPHA_NUM not in _ALPHA_REGISTRY:
    raise KeyError(f'Alpha{ALPHA_NUM} 未注册')
info = _ALPHA_REGISTRY[ALPHA_NUM]
print(f'=== Alpha{ALPHA_NUM} ===')
print(f'函数: {info["name"]}')
print(f'分类: {info.get("category", "price-volume")}')
print(f'描述: {info["desc"]}')
print('\nDocstring:')
print(info['func'].__doc__ or '(无)')

## 2. 生成数据 + 算因子

用合成 100 票 × 5 年 panel。Alpha22 等需要长窗口的因子需要 ≥ 250 天数据。

In [ ]:
panel = generate_synthetic_panel(n_stocks=100, years=5, seed=2026)
factor = compute_alpha(ALPHA_NUM, panel)
factor_pp = preprocess_pipeline(factor)
print(f'panel: {panel["close"].shape}, factor non-NaN ratio: {factor.notna().sum().sum() / factor.size:.1%}')
print('\n最新 10 天 × 前 5 票 (preprocessed):')
factor_pp.iloc[-10:, :5]

## 3. 滚动 IC 时间序列

In [ ]:
fwd_ret = compute_forward_return(panel['close'], horizon=HORIZON)
ic_daily = daily_ic(factor_pp, fwd_ret, method='spearman')
ic_60d = ic_daily.rolling(60, min_periods=20).mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(ic_daily.index, ic_daily.values, alpha=0.3, color='gray', label='日 IC')
ax.plot(ic_60d.index, ic_60d.values, color='C0', linewidth=2, label='60 日滚动均值')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_title(f'Alpha{ALPHA_NUM} — 日 IC + 60 日均值 (vs {HORIZON} 日前瞻收益)')
ax.set_ylabel('IC (Spearman)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
print(f'IC mean = {ic_daily.mean():.4f}, std = {ic_daily.std():.4f}, ICIR = {ic_daily.mean()/ic_daily.std()*np.sqrt(252):.2f}')
print(f'正胜率 = {(ic_daily > 0).sum() / ic_daily.notna().sum():.1%}')

## 4. IC 自相关 — 信号衰减速度

In [ ]:
lags = range(1, 21)
autocorr = [ic_daily.autocorr(lag=l) for l in lags]
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(lags, autocorr, color='steelblue', alpha=0.8)
ax.axhline(0, color='red', linestyle='--', alpha=0.4)
ax.set_xticks(lags)
ax.set_xlabel('Lag (天)')
ax.set_ylabel('IC autocorrelation')
ax.set_title(f'Alpha{ALPHA_NUM} — IC 自相关 (>0 说明信号在持续，衰减慢)')
ax.grid(alpha=0.3);

## 5. Top decile 累计超额 PnL

In [ ]:
excess = top_decile_excess(factor_pp, fwd_ret, decile=10)
cum_excess = (1 + excess.fillna(0)).cumprod()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cum_excess.index, cum_excess.values, color='C2', linewidth=1.6)
ax.axhline(1, color='black', alpha=0.4)
ax.set_title(f'Alpha{ALPHA_NUM} — Top 10% 等权 vs 等权全市场累计超额 (重叠 {HORIZON} 日)')
ax.set_ylabel('累计超额')
ax.grid(alpha=0.3)
print(f'总超额 = {cum_excess.iloc[-1] - 1:.2%}')

## 6. 横截面分布 — 最新一日

In [ ]:
last_row = factor_pp.iloc[-1].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(last_row.values, bins=30, color='C0', alpha=0.8)
axes[0].set_title(f'最新一日 ({factor_pp.index[-1].date()}) — 因子值分布')
axes[0].set_xlabel('factor value (z-scored)')
axes[0].grid(alpha=0.3)
axes[1].boxplot([factor_pp.iloc[-30 + i].dropna().values for i in range(0, 30, 5)],
                labels=[(factor_pp.index[-30+i].date().strftime('%m-%d')) for i in range(0, 30, 5)])
axes[1].set_title('最近 30 日横截面分布 (每 5 日采样)')
axes[1].set_ylabel('factor value')
axes[1].grid(alpha=0.3)
plt.tight_layout()

## 7. NaN 率随时间

In [ ]:
nan_ratio = factor.isna().sum(axis=1) / factor.shape[1]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(nan_ratio.index, nan_ratio.values, color='C3', linewidth=1.2)
ax.set_title(f'Alpha{ALPHA_NUM} — 每日 NaN 占比 (前期是窗口暖机，正常)')
ax.set_ylabel('NaN ratio')
ax.grid(alpha=0.3)
print(f'总 NaN 比例: {factor.isna().sum().sum() / factor.size:.2%}')

## 8. 结论模板

- IC 稳定性: ___
- 信号衰减: lag 1-3 自相关 ___，建议 ___
- Top decile 超额: 总 ___% / 月度胜率 ___
- 数据质量: NaN 暖机后稳定在 ___% 以下
- 与论文预期: ___